In [14]:
from pyspark.sql import SparkSession

In [15]:
spark=SparkSession.builder.appName('json').getOrCreate();

In [23]:
df=spark.read.format('json').load('file:///home/unik/OReily/2015-summary.json',header=True,inferSchema=True)

# COLUMNS

In [24]:
df.show()

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Romania|   15|
|       United States|            Croatia|    1|
|       United States|            Ireland|  344|
|               Egypt|      United States|   15|
|       United States|              India|   62|
|       United States|          Singapore|    1|
|       United States|            Grenada|   62|
|          Costa Rica|      United States|  588|
|             Senegal|      United States|   40|
|             Moldova|      United States|    1|
|       United States|       Sint Maarten|  325|
|       United States|   Marshall Islands|   39|
|              Guyana|      United States|   64|
|               Malta|      United States|    1|
|            Anguilla|      United States|   41|
|             Bolivia|      United States|   30|
|       United States|           Paraguay|    6|
|             Algeri

In [27]:
from pyspark.sql.functions import col
df.select(col('count')).show()

+-----+
|count|
+-----+
|   15|
|    1|
|  344|
|   15|
|   62|
|    1|
|   62|
|  588|
|   40|
|    1|
|  325|
|   39|
|   64|
|    1|
|   41|
|   30|
|    6|
|    4|
|  230|
|    1|
+-----+
only showing top 20 rows



In [31]:
spark.read.format('json').load('file:///home/unik/OReily/2015-summary.json').columns

['DEST_COUNTRY_NAME', 'ORIGIN_COUNTRY_NAME', 'count']

# RECORDS AND ROWS

In [32]:
df.first()

Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count=15)

## CREATING ROWS
    -You can create rows by manually instantiating a Row object with the values that belong in each column.
    
    -If you create a Row manually, you must specify the values in the same order as the schema of the DataFrame to which they might be appended

In [33]:
from pyspark.sql import Row
myRow=Row('Hello',None,2,False)

In [34]:
myRow[0]

'Hello'

# DATAFRAME TRANSFORMATIONS
    -We can add rows or columns

    -We can remove rows or columns

    -We can transform a row into a column (or vice versa)

    -We can change the order of rows based on the values in columns

<img src='https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781491912201/files/assets/spdg_0502.png' width='500'>

## CREATING DATAFRAMES

In [39]:
from pyspark.sql import Row
from pyspark.sql.types import StructField, StructType, StringType, LongType
myManualSchema = StructType([
  StructField("some", StringType(), True),
  StructField("col", StringType(), True),
  StructField("names", LongType(), False)
])
myRow = Row("Hello", None, 1)
myDf = spark.createDataFrame([myRow], myManualSchema)
myDf.show()

+-----+----+-----+
| some| col|names|
+-----+----+-----+
|Hello|NULL|    1|
+-----+----+-----+



## select AND selectExpr 

In [42]:
df.select('DEST_COUNTRY_NAME').show(2)

+-----------------+
|DEST_COUNTRY_NAME|
+-----------------+
|    United States|
|    United States|
+-----------------+
only showing top 2 rows



In [43]:
df.select("DEST_COUNTRY_NAME","ORIGIN_COUNTRY_NAME").show(2)

+-----------------+-------------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|
+-----------------+-------------------+
|    United States|            Romania|
|    United States|            Croatia|
+-----------------+-------------------+
only showing top 2 rows



In [50]:
#DIFFERENT WAYS OF SELECTING COLUMNS
from pyspark.sql.functions import expr, col, column
df.select(
    expr("DEST_COUNTRY_NAME"),
    col("DEST_COUNTRY_NAME"),
    column("DEST_COUNTRY_NAME"))\
  .show(2)

+-----------------+-----------------+-----------------+
|DEST_COUNTRY_NAME|DEST_COUNTRY_NAME|DEST_COUNTRY_NAME|
+-----------------+-----------------+-----------------+
|    United States|    United States|    United States|
|    United States|    United States|    United States|
+-----------------+-----------------+-----------------+
only showing top 2 rows



In [51]:
# USE OF AS AND ALIAS
# df.select(expr("DEST_COUNTRY_NAME AS destination")).show(2)
df.select(expr("DEST_COUNTRY_NAME as destination").alias("DEST_COUNTRY_NAME"))\
  .show(2)

+-----------------+
|DEST_COUNTRY_NAME|
+-----------------+
|    United States|
|    United States|
+-----------------+
only showing top 2 rows



In [52]:
# BY USING SHORTHAND selectExpr INSTEAD OF df.select(expr())
df.selectExpr("DEST_COUNTRY_NAME as newColumnName", "DEST_COUNTRY_NAME").show(2)

+-------------+-----------------+
|newColumnName|DEST_COUNTRY_NAME|
+-------------+-----------------+
|United States|    United States|
|United States|    United States|
+-------------+-----------------+
only showing top 2 rows



In [58]:
from pyspark.sql.functions import col
df.selectExpr(
    "*",
    "(DEST_COUNTRY_NAME = ORIGIN_COUNTRY_NAME) as withinCountry"
).where(col("withinCountry") == True) \
 .show()

+-----------------+-------------------+------+-------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME| count|withinCountry|
+-----------------+-------------------+------+-------------+
|    United States|      United States|370002|         true|
+-----------------+-------------------+------+-------------+



In [60]:
# With select expression, we can also specify aggregations over the entire DataFrame by taking advantage of the functions that we have.
df.selectExpr("avg(count)", "count(distinct(DEST_COUNTRY_NAME))").show(2)

+-----------+---------------------------------+
| avg(count)|count(DISTINCT DEST_COUNTRY_NAME)|
+-----------+---------------------------------+
|1770.765625|                              132|
+-----------+---------------------------------+



## ADDING COLUMNS
    -There’s also a more formal way of adding a new column to a DataFrame, and that’s by using the withColumn method on our DataFrame.

In [61]:
df.withColumn("numberOne", lit(1)).show(2)

+-----------------+-------------------+-----+---------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|numberOne|
+-----------------+-------------------+-----+---------+
|    United States|            Romania|   15|        1|
|    United States|            Croatia|    1|        1|
+-----------------+-------------------+-----+---------+
only showing top 2 rows



In [64]:
#We set a Boolean flag for when the origin country is the same as the destination country|
df.withColumn('withinCountry',expr('ORIGIN_COUNTRY_NAME==DEST_COUNTRY_NAME')).show(2)

+-----------------+-------------------+-----+-------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|withinCountry|
+-----------------+-------------------+-----+-------------+
|    United States|            Romania|   15|        false|
|    United States|            Croatia|    1|        false|
+-----------------+-------------------+-----+-------------+
only showing top 2 rows



## RENAMING COLUMNS 
    -Although we can rename a column in the manner that we just described, another alternative is to use the withColumnRenamed method.

In [66]:
df.withColumnRenamed("DEST_COUNTRY_NAME",'dest').columns

['dest', 'ORIGIN_COUNTRY_NAME', 'count']

## REMOVING COLUMNS

In [68]:
df.drop('ORIGIN_COUNTRY_NAME').columns

['DEST_COUNTRY_NAME', 'count']

## CHANGING COLUMNS TYPE USING (cast)

In [70]:
df.withColumn('count2',col('count').cast('long')).columns

['DEST_COUNTRY_NAME', 'ORIGIN_COUNTRY_NAME', 'count', 'count2']

## FILTERING ROWS
    -To filter rows, we create an expression that evaluates to true or false
    
    -There are two methods to perform this operation: you can use where or filter and they both will perform the same operation and accept the same argument types when used with DataFrames.

In [78]:
# BOTH PERFORM SAME OPERTAIONS
df.filter(col('count')>5).show(5)
# df.where(col('count')>5).show(5)

#This opertaion has used select to filter only destination names
# df.filter(col('count')>5).select('DEST_COUNTRY_NAME').show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
|    United States|              India|   62|
|    United States|            Grenada|   62|
+-----------------+-------------------+-----+
only showing top 5 rows



In [81]:
#Multiple filtering
df.where(col("count")<2).where(col("ORIGIN_COUNTRY_NAME")!= "Croatia").show(2)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|          Singapore|    1|
|          Moldova|      United States|    1|
+-----------------+-------------------+-----+
only showing top 2 rows



## GETTING UNIQUE ROWS
    -The way we do this is by using the distinct method on a DataFrame, which allows us to deduplicate any rows that are in that DataFrame. 

In [83]:
df.select(col('DEST_COUNTRY_NAME'),col('ORIGIN_COUNTRY_NAME')).distinct().count()

256

In [84]:
df.select('ORIGIN_COUNTRY_NAME').distinct().count()

125

## RANDOM SAMPLES
    -Sometimes, you might just want to sample some random records from your DataFrame. You can do this by using the sample method on a DataFrame, which makes it possible for you to specify a fraction of rows to extract from a DataFrame and whether you’d like to sample with or without replacement

In [85]:
seed=5
withReplacement=False
fraction=0.5
df.sample(withReplacement,fraction,seed).count()

138

## RANDOM SPLITS


In [92]:
dataframes=df.randomSplit([0.25,0.75],seed)
dataframes[0].count()>dataframes[1].count()

False

## CONCATENATING AND APPENDING ROWS (UNION)
    -To append to a DataFrame, you must union the original DataFrame along with the new DataFrame
    
    -To union two DataFrames, you must be sure that they have the same schema and number of columns; otherwise, the union will fail.

In [98]:
schema=df.schema
newRows=[
    Row("New Country",'Other Country',5),
    Row('New Counry 2','Other Country 3',1)
]
parallelizedRows=spark.sparkContext.parallelize(newRows)
newDF=spark.createDataFrame(parallelizedRows,schema)

In [99]:
df.union(newDF)\
  .where("count = 1")\
  .where(col("ORIGIN_COUNTRY_NAME") != "United States")\
  .show()

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Croatia|    1|
|    United States|          Singapore|    1|
|    United States|          Gibraltar|    1|
|    United States|             Cyprus|    1|
|    United States|            Estonia|    1|
|    United States|          Lithuania|    1|
|    United States|           Bulgaria|    1|
|    United States|            Georgia|    1|
|    United States|            Bahrain|    1|
|    United States|   Papua New Guinea|    1|
|    United States|         Montenegro|    1|
|    United States|            Namibia|    1|
|     New Counry 2|    Other Country 3|    1|
+-----------------+-------------------+-----+



## SORTING ROWS
    -There are two equivalent operations to do this sort and orderBy that work the exact same way.
    
    -They accept both column expressions and strings as well as multiple columns.

In [104]:
# df.sort('count').show(5)
# df.orderBy('count','DEST_COUNTRY_NAME').show(5)
df.orderBy(col('count').desc(),col('DEST_COUNTRY_NAME').asc()).show(5)

+-----------------+-------------------+------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME| count|
+-----------------+-------------------+------+
|    United States|      United States|370002|
|    United States|             Canada|  8483|
|           Canada|      United States|  8399|
|    United States|             Mexico|  7187|
|           Mexico|      United States|  7140|
+-----------------+-------------------+------+
only showing top 5 rows



# LIMIT

In [105]:
df.limit(5).show()

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
|    United States|              India|   62|
+-----------------+-------------------+-----+



# REPARTITION AND COALESCE
    -Another important optimization opportunity is to partition the data according to some frequently filtered columns, which control the physical layout of data across the cluster including the partitioning scheme and the number of partitions.
    
    -Repartition will incur a full shuffle of the data, regardless of whether one is necessary. This means that you should typically only repartition when the future number of partitions is greater than your current number of partitions or when you are looking to partition by a set of columns

In [107]:
df.repartition(col("DEST_COUNTRY_NAME"))

DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: bigint]

In [108]:
df.repartition(5,col('DEST_COUNTRY_NAME'))

DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: bigint]

    -Coalesce, on the other hand, will not incur a full shuffle and will try to combine partitions. This operation will shuffle your data into five partitions based on the destination country name, and then coalesce them (without a full shuffle)

In [111]:
df.repartition(5, col("DEST_COUNTRY_NAME")).coalesce(2)

DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: bigint]